In [1]:
import os
import glob
import warnings
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel

In [2]:
# =========================================================
# 0) 환경 설정 및 경로 정의
# =========================================================
warnings.filterwarnings("ignore")

project_dir = "/home/remote/Ai_Capstone_Project"
model_path  = os.path.join(project_dir, "polyglot-ko-3.8B")
data_file   = os.path.join(project_dir, "data_singleline.jsonl")
finetune_dir = os.path.join(project_dir, "Model_part2", "Fine-tuning-LoRA")
merged_dir   = os.path.join(project_dir, "Model_part2", "Merged_model")

os.makedirs(finetune_dir, exist_ok=True)
os.makedirs(merged_dir, exist_ok=True)

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    load_in_8bit=True,
    device_map="auto",
    use_safetensors=True  # safetensors 샤드를 자동으로 로드
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [4]:
# 셀 5: LoRA 어댑터 설정 및 파라미터 수 확인
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["attention.dense"],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
# LoRA 파라미터 수
total_params = sum(p.numel() for p in model.parameters())
lora_params = sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and 'lora_' in n)
print(f"모델 전체 파라미터 수: {total_params}")
print(f"LoRA 어댑터 파라미터 수: {lora_params} ({lora_params/total_params*100:.2f}% 비율)")

모델 전체 파라미터 수: 3811547136
LoRA 어댑터 파라미터 수: 1572864 (0.04% 비율)


In [5]:
# =========================================================
# 3) Streaming 모드로 데이터셋 불러오기
# =========================================================
# streaming=True를 주면 IterableDataset을 반환
dataset_stream = load_dataset(
    "json",
    data_files={"train": data_file},
    split="train",
    streaming=True,
    use_auth_token=False  # (필요시 HuggingFace 토큰)
)

In [6]:
# =========================================================
# 4) IterableDataset을 위한 토크나이징 함수
#    - 배치 단위가 아니라, 한 샘플씩(tokenizer 내에서 자동 패딩X)을 처리
# =========================================================
def streaming_tokenize(example):
    """
    example: {"field1": ..., "field2": ..., ...} 형태의 dict
    1) prompt/response 필드가 있으면 "<|sep|>" 결합
    2) instruction/output 필드가 있으면 "<|sep|>" 결합
    3) 그 외에는 모든 값을 공백으로 연결
    """
    if "prompt" in example and "response" in example:
        text = example["prompt"] + "<|sep|>" + example["response"]
    elif "instruction" in example and "output" in example:
        text = example["instruction"] + "<|sep|>" + example["output"]
    elif "text" in example:
        text = example["text"]
    else:
        text = " ".join([str(v) for v in example.values()])

    # 토크나이저에 바로 인코딩(길이 초과 시 잘라내기)
    # return dict 형식 (Trainer가 받아들이도록)
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=512,         # (원하는 최대 길이)
        return_tensors="pt",    # PyTorch tensor 반환
    )
    # 여기서는 "input_ids"와 "attention_mask"만 필요하므로 그대로 반환
    return {
        "input_ids": tokenized["input_ids"].squeeze(0),       # 차원: (seq_len,)
        "attention_mask": tokenized["attention_mask"].squeeze(0)
    }

In [7]:
# =========================================================
# 5) IterableDataset에 토크나이징 적용
#    - .map()을 호출하면 새로운 IterableDataset이 생성됨
# =========================================================
# batched=False 로 설정해 한 예제를 한 번에 처리
tokenized_stream = dataset_stream.map(
    streaming_tokenize,
    batched=False
)


In [8]:
# =========================================================
# 6) Trainer 설정
#    - IterableDataset을 그대로 train_dataset으로 넣을 수 있음
#    - num_train_epochs 대신 max_steps로 스텝 수 지정
# =========================================================
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir=os.path.join(finetune_dir, "checkpoints"),
    overwrite_output_dir=True,
    do_train=True,
    max_steps=2000,                 # 총 몇 스텝 학습할지 설정
    per_device_train_batch_size=4,  # 실제 GPU VRAM 상황에 맞춰 조정
    gradient_accumulation_steps=1,
    logging_strategy="steps",
    logging_steps=100,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    report_to="none",
    logging_dir=os.path.join(finetune_dir, "logs"),
    fp16=True,
    optim="paged_adamw_8bit",
    disable_tqdm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_stream,  # IterableDataset 그대로 전달
    data_collator=data_collator,
)


In [9]:
# =========================================================
# 7) 학습 실행
# =========================================================
trainer.train()


You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
100,2.414300
200,0.425900
300,0.663300
400,0.287700
500,0.232800
600,0.451800
700,0.234200
800,0.183100
900,0.385500
1000,0.178500


TrainOutput(global_step=2000, training_loss=0.45324073696136474, metrics={'train_runtime': 481.6872, 'train_samples_per_second': 16.608, 'train_steps_per_second': 4.152, 'total_flos': 1.3775045087821824e+16, 'train_loss': 0.45324073696136474, 'epoch': 1.0})

In [10]:
# =========================================================
# 8) 학습된 LoRA 가중치 저장 + 메모리 해제
# =========================================================
trainer.model.save_pretrained(finetune_dir)
del model, trainer
torch.cuda.empty_cache()


In [11]:
# =========================================================
# 9) LoRA 병합 및 최종 모델 저장
# =========================================================
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map={"": "cpu"}
)
lora_model = PeftModel.from_pretrained(base_model, finetune_dir)
merged_model = lora_model.merge_and_unload()
merged_model.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)

print("✅ Streaming 모드로 LoRA 파인튜닝 + 병합 완료")
print(f"  • Fine-tuning 체크포인트: {finetune_dir}")
print(f"  • 최종 병합 모델: {merged_dir}")

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Streaming 모드로 LoRA 파인튜닝 + 병합 완료
  • Fine-tuning 체크포인트: /home/remote/Ai_Capstone_Project/Model_part2/Fine-tuning-LoRA
  • 최종 병합 모델: /home/remote/Ai_Capstone_Project/Model_part2/Merged_model
